# Roteiro da Primeira Implementação

1. Introdução
2. Importação das bibliotecas
3. Configuração da API
4. Função da coleta de dados
5. Execução da coleta
6. Extração das métricas
7. Conversão para dataframe e normalização dos dados
8. Visualização dos dados

## Introdução
### Coleta de Dados — RIPE Atlas

Este notebook implementa a primeira etapa do projeto de predição
de falhas de rede.

O objetivo desta etapa é realizar a coleta de dados de medições
de rede utilizando a API do RIPE Atlas.

Os dados coletados serão posteriormente tratados e utilizados
como entrada para um modelo baseado em árvore de decisão.

## Importação das bibliotecas

In [ ]:
import requests
import pandas as pd
import datetime as dt

## Configuração da API

In [ ]:
# gera dois datetimes, sendo um deles com um segundo de diferença
# para retornar dados atualizados da API no intervalo de um segundo

current_datetime = dt.datetime.today()

start_datetime = current_datetime - dt.timedelta(minutes=5)
stop_datetime = current_datetime

BASE_URL = 'https://atlas.ripe.net/api/v2'
MEASUREMENT_ID = 1001

# sempre coleta dados atualizados, levando em conta a hora atual
START_TIME = start_datetime.strftime('%Y-%m-%dT%H:%M:%S')
STOP_TIME = stop_datetime.strftime('%Y-%m-%dT%H:%M:%S')
API_URL = f'{BASE_URL}/measurements/{MEASUREMENT_ID}/results/'

# parametros da requisição
params = {
    'start': START_TIME,
    'stop': STOP_TIME
}

## Função da Coleta de Dados

In [ ]:
def collect_data(url: str, params: dict):
    response = requests.get(url, params=params)

    response.raise_for_status()

    return response.json()

## Execução da coleta


In [ ]:
data = collect_data(API_URL, params)

print(data)

[{'fw': 5080, 'mver': '2.6.2', 'lts': 1632363, 'dst_name': '193.0.14.129', 'af': 4, 'dst_addr': '193.0.14.129', 'src_addr': '192.168.2.200', 'proto': 'ICMP', 'ttl': 57, 'size': 32, 'result': [{'rtt': 20.069782}, {'rtt': 19.696431}, {'rtt': 19.857497}], 'dup': 0, 'rcvd': 3, 'sent': 3, 'min': 19.696431, 'max': 20.069782, 'avg': 19.874570000000002, 'msm_id': 1001, 'prb_id': 1000023, 'timestamp': 1789083358, 'msm_name': 'Ping', 'from': '159.146.31.90', 'type': 'ping', 'step': 240, 'stored_timestamp': 1789083423}, {'fw': 5100, 'mver': '2.6.4', 'lts': 16, 'dst_name': '193.0.14.129', 'af': 4, 'dst_addr': '193.0.14.129', 'src_addr': '85.209.51.37', 'proto': 'ICMP', 'size': 32, 'result': [{'error': 'sendto failed: Operation not permitted'}, {'error': 'sendto failed: Operation not permitted'}, {'error': 'sendto failed: Operation not permitted'}], 'dup': 0, 'rcvd': 0, 'sent': 0, 'min': -1, 'max': -1, 'avg': -1, 'msm_id': 1001, 'prb_id': 1000030, 'timestamp': 1789083336, 'msm_name': 'Ping', 'from'

## Função para Extração das Métricas Relevantes

In [ ]:
def extract_relevant_data(data):
    records = []

    for item in data:
        record = {
            'timestamp': item.get('timestamp'),
            'ip': item.get('src_addr'),
            'latencia_ms': item.get('avg'),
            'pacotes_enviados': item.get('sent'),
            'pacotes_recebidos': item.get('rcvd'),
        }

        records.append(record)

    return records

## Extração das métricas

In [ ]:
relevant_data = extract_relevant_data(data)

print(relevant_data)

[{'timestamp': 1789083358, 'ip': '192.168.2.200', 'latencia_ms': 19.874570000000002, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083336, 'ip': '85.209.51.37', 'latencia_ms': -1, 'pacotes_enviados': 0, 'pacotes_recebidos': 0}, {'timestamp': 1789083359, 'ip': '192.168.0.14', 'latencia_ms': 123.50172766666667, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083351, 'ip': '89.189.191.217', 'latencia_ms': 0.4482296666666667, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083336, 'ip': '188.75.184.76', 'latencia_ms': 8.879265, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083339, 'ip': '10.50.84.12', 'latencia_ms': 8.518683999999999, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083352, 'ip': '147.78.245.129', 'latencia_ms': 1.3371203333333332, 'pacotes_enviados': 3, 'pacotes_recebidos': 3}, {'timestamp': 1789083320, 'ip': '37.35.109.50', 'latencia_ms': 5.304711666666667, 'pacotes_enviados': 3,

## Conversão para DataFrame e Normalização dos Dados

In [ ]:
def normalize_relevant_data(data):
    data = pd.DataFrame(data)

    data['timestamp'] = data['timestamp'].astype('timestamp[s][pyarrow]')

    return data

normalized_data = normalize_relevant_data(relevant_data)

## Visualização dos dados

In [ ]:
display(normalized_data)

,timestamp,ip,latencia_ms,pacotes_enviados,pacotes_recebidos
0,2026-09-10 23:35:58,192.168.2.200,19.874570,3,3
1,2026-09-10 23:35:36,85.209.51.37,-1.000000,0,0
2,2026-09-10 23:35:59,192.168.0.14,123.501728,3,3
3,2026-09-10 23:35:51,89.189.191.217,0.448230,3,3
4,2026-09-10 23:35:36,188.75.184.76,8.879265,3,3
...,...,...,...,...,...
13407,2026-09-10 23:40:01,192.168.0.230,11.271138,3,3
13408,2026-09-10 23:40:01,199.47.174.226,5.539125,3,3
13409,2026-09-10 23:40:02,192.168.2.160,10.135083,3,3
13410,2026-09-10 23:40:07,192.168.18.239,45.036262,3,3
